# Analyzing Recommendation System Performance Across Model Versions

- [View Solution Notebook](./solution.html)
- [View Project Page](https://www.codecademy.com/content-items/13f970fd92fa48db8d99b770964b037e)


## Background
You're an AI Engineer at a growing e-commerce company that has been experimenting with AI-powered product recommendations. Over the past 10 months, the data science team deployed three versions of their recommendation model:

* v1.0 - The baseline model (cheapest to run)
* v1.1 - An improved version with better accuracy (moderate cost)
* v2.0 - The latest model using advanced techniques (most expensive)

Each newer version costs more to operate due to increased computational requirements, but the team hopes these investments will pay off through higher sales. Your job is to determine which model actually delivers the best return on investment.

## The Dataset
This project uses synthetically generated data that simulates real e-commerce behavior. The dataset comprises 10 months of customer interactions, featuring realistic patterns in purchasing behavior, demographic distributions, and recommendation outcomes. While the data is synthetic, it reflects the types of challenges and patterns you'd encounter in production AI systems.

### Key Metrics You'll Analyze
1. Conversion Rate: The percentage of customer interactions that result in a purchase. For example, if 100 customers view products and 8 make purchases, that's an 8% conversion rate. This is your primary success metric—did recommendations actually drive more sales?
2. ROI (Return on Investment): How much profit you generate for every dollar spent on the AI system. An ROI of 150% means for every `$1` spent running the model, you earn `$2.50` back (`$1` returned + `$1.50` profit). This tells you whether the AI investment is financially worthwhile.
3. Profit: Revenue from sales minus all costs (both product costs and AI inference costs). This is your bottom line—the actual money made after accounting for everything.

Your analysis will directly influence a major business decision: should the company stick with the cheaper v1.0, upgrade to the expensive v2.0, or adopt a smart routing strategy using different models for different customers?

Let's find out!

## Setup and Initial Data Exploration

**Task 1:** Import pandas as `pd`, numpy as `np`, and from `scipy.stats` import `chi2_contingency` for statistical testing.

In [1]:
# Your code here
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

**Task 2:** Load the CSV file `ecommerce_recommendation_events.csv` into a DataFrame called `events`. Display the first 10 rows and print the shape to understand the dataset size.

In [2]:
# Your code here
events = pd.read_csv('./ecommerce_recommendation_events.csv')
print(events.shape)
events.head(10)

(50000, 18)


,timestamp,customer_id,email,phone,product_id,event_type,recommended,model_version,age,region,customer_segment,product_category,price_tier,revenue,product_cost,inference_cost,total_cost,profit
0,1704092618,4464,sophia.anderson4464@email.com,+1-636-599-5625,408,purchase,0,v1.0,25,South,budget,electronics,high,858.29,415.88,0.000,415.880,442.410
1,1704093226,3079,amelia.gonzalez3079@inbox.com,+1-313-997-8700,51,view,0,v1.0,25,South,premium,electronics,medium,0.00,0.00,0.000,0.000,0.000
2,1704093327,4757,mason.davis4757@inbox.com,+1-556-615-7307,296,view,0,v1.0,18,North,regular,clothing,high,0.00,0.00,0.000,0.000,0.000
3,1704094155,938,isabella.rodriguez938@email.com,+1-345-911-8113,33,view,0,v1.0,55,North,budget,sports,low,0.00,0.00,0.000,0.000,0.000
4,1704094174,2128,ava.jackson2128@email.com,+1-482-307-4871,927,view,1,v1.0,25,North,premium,sports,low,0.00,0.00,0.003,0.003,-0.003
5,1704094505,3757,ethan.moore3757@email.com,+1-654-968-2040,414,addtocart,1,v1.0,25,North,regular,clothing,low,0.00,0.00,0.003,0.003,-0.003
6,1704095017,1331,mason.garcia1331@mail.com,+1-580-491-7965,838,view,1,v1.0,45,North,premium,home,low,0.00,0.00,0.003,0.003,-0.003
7,1704096951,3512,henry.martinez3512@inbox.com,+1-746-633-6045,306,addtocart,0,v1.0,18,East,premium,home,high,0.00,0.00,0.000,0.000,0.000
8,1704097147,3903,alexander.davis3903@mail.com,+1-511-338-5713,39,view,0,v1.0,55,East,regular,books,medium,0.00,0.00,0.000,0.000,0.000
9,1704097404,595,william.moore595@inbox.com,+1-897-735-5477,812,purchase,1,v1.0,55,South,premium,books,low,18.91,10.37,0.003,10.373,8.537


**Task 3:** Display the column names and data types using `.dtypes` and print the list of all column names.

In [3]:
# Your code here
print(events.dtypes)
print(events.columns.tolist())

timestamp             int64
customer_id           int64
email                   str
phone                   str
product_id            int64
event_type              str
recommended           int64
model_version           str
age                   int64
region                  str
customer_segment        str
product_category        str
price_tier              str
revenue             float64
product_cost        float64
inference_cost      float64
total_cost          float64
profit              float64
dtype: object
['timestamp', 'customer_id', 'email', 'phone', 'product_id', 'event_type', 'recommended', 'model_version', 'age', 'region', 'customer_segment', 'product_category', 'price_tier', 'revenue', 'product_cost', 'inference_cost', 'total_cost', 'profit']


**Task 4:** Create a new column called `converted` that equals `True` when `event_type` is `'purchase'` and `False` otherwise. Then print the total number of events, total purchases, and overall conversion rate.

In [4]:
# Your code here
events['converted'] = events['event_type'] == 'purchase'
total_number_of_events = events['event_type'].count()
total_purchases = events['converted'].sum()
conversion_rate = (total_purchases / total_number_of_events)*100
print(total_number_of_events)
print(total_purchases)
print(f"{conversion_rate:.2f}%")

50000
3225
6.45%


## Financial Performance Analysis

**Task 5:** Calculate overall financial metrics from converted events only: sum of `revenue`, sum of `product_cost`, sum of `total_cost`, and sum of `profit`. Also, calculate the total `inference_cost` across ALL events. Finally, calculate the average profit per conversion.


In [5]:
# Your code here
total_revenue = 0
total_product_cost = 0
total_cost = 0
total_profit = 0

if events['converted'].any():
    total_revenue = events.loc[events['converted'], 'revenue'].sum()
    total_product_cost = events.loc[events['converted'], 'product_cost'].sum()
    total_cost = events.loc[events['converted'], 'total_cost'].sum()
    total_profit = events.loc[events['converted'], 'profit'].sum()

total_inference_cost = events['inference_cost'].sum()
average_profit_per_conversion = total_profit / total_purchases if total_purchases > 0 else 0  # type: ignore

print(f"Total Revenue: {total_revenue}")
print(f"Total Product Cost: {total_product_cost}")
print(f"Total Cost: {total_cost}")
print(f"Total Profit: {total_profit}")
print(f"Total Inference Cost: {total_inference_cost}")
print(f"Average Profit per Conversion: {average_profit_per_conversion}")

Total Revenue: 580316.6599999999
Total Product Cost: 273011.92000000004
Total Cost: 273021.99100000004
Total Profit: 307294.669
Total Inference Cost: 106.65
Average Profit per Conversion: 95.28516868217054


**Task 6:** Group by the `recommended` column (0 or 1) and calculate the mean of both `converted` and `profit` columns. Calculate lift for both metrics by dividing the recommended values by the non-recommended values.


In [6]:
# Your code here

profit_conversion_by_recommended = events.groupby('recommended').agg({
    'converted': 'mean',
    'profit': 'mean',
})

conversion_rates = profit_conversion_by_recommended['converted'].reindex([0, 1]).to_numpy(dtype=np.float64)
mean_profits = profit_conversion_by_recommended['profit'].reindex([0, 1]).to_numpy(dtype=np.float64)

converted_lift = conversion_rates[1] / conversion_rates[0]
profit_lift = mean_profits[1] / mean_profits[0]

print(f"Conversion Lift: {converted_lift}")
print(f"Profit Lift: {profit_lift}")

Conversion Lift: 2.4526647142458478
Profit Lift: 2.843268285979508


## Model Version Performance Comparison

**Task 7:** Group by `model_version` and calculate: count of events, mean of `converted`, sum of `revenue`, sum of `inference_cost`, sum of `product_cost`, and sum of `profit`. 

Then calculate two different ROI metrics to understand both the AI-specific return and overall business profitability:
1. AI ROI - Measures return on the AI investment specifically (profit / inference_cost × 100)
2. Overall ROI - Measures total business profitability, including all costs (profit / total_costs × 100)

In [7]:
# Your code here
events_by_model_version = events.groupby('model_version').agg({
    'event_type': 'count',
    'converted': 'mean',
    'revenue': 'sum',
    'inference_cost': 'sum',
    'product_cost': 'sum',
    'profit': 'sum'
    })

AI_ROI = events_by_model_version['profit'] / (events_by_model_version['inference_cost']) * 100
Overall_ROI = (events_by_model_version['profit'].sum() /(events_by_model_version['inference_cost'].sum() + events_by_model_version['product_cost'].sum())) * 100

print(f"AI ROI by Model Version:\n{AI_ROI}")
print(f"Overall ROI: {Overall_ROI}")

AI ROI by Model Version:
model_version
v1.0    405110.269902
v1.1    320073.174476
v2.0    228166.720085
dtype: float64
Overall ROI: 112.47792121934437


**Task 8:** Filter events to only rows where `recommended` equals 1. Group by `model_version` and calculate the mean conversion rate and the average profit per conversion (profit only for converted events).


In [8]:
# Your code here
recommended_events = events[events['recommended'] == 1]

mean_conversion_rate = recommended_events.groupby('model_version')['converted'].mean()
average_profit_per_conversion = (recommended_events[recommended_events['converted'] == 1].groupby('model_version')['profit'].mean())

print(f"Mean Conversion Rate by Model Version:\n{mean_conversion_rate}")
print(f"Average Profit per Conversion by Model Version:\n{average_profit_per_conversion}")

Mean Conversion Rate by Model Version:
model_version
v1.0    0.087548
v1.1    0.096916
v2.0    0.095620
Name: converted, dtype: float64
Average Profit per Conversion by Model Version:
model_version
v1.0     96.143344
v1.1    101.308080
v2.0    100.426402
Name: profit, dtype: float64


## Regional Bias Detection

**Task 9:** Calculate the overall recommendation rate, then group by `region` to get recommendation rates for each region. Add columns showing the difference from average and a boolean `potential_bias` column that's True when the absolute difference exceeds 0.10 (10%).


In [9]:
# Your code here
overall_recommendation_rate = events['recommended'].mean()

region_rates = (
    events.groupby('region')['recommended']
    .mean()
    .to_frame(name='region_recommendation_rate')
)

region_rates['difference_from_average'] = (
    region_rates['region_recommendation_rate']
    - overall_recommendation_rate
)

region_rates['potential_bias'] = (
    region_rates['difference_from_average'].abs() > 0.10
)

print(f"Region Recommendation Rates:\n{region_rates}")
print(f"Overall Recommendation Rate: {overall_recommendation_rate}")
print(f"Regions with Potential Bias:\n{region_rates[region_rates['potential_bias']]}")

Region Recommendation Rates:
        region_recommendation_rate  difference_from_average  potential_bias
region                                                                     
East                      0.515009                 0.041149           False
North                     0.514831                 0.040971           False
South                     0.425563                -0.048297           False
West                      0.427985                -0.045875           False
Overall Recommendation Rate: 0.47386
Regions with Potential Bias:
Empty DataFrame
Columns: [region_recommendation_rate, difference_from_average, potential_bias]
Index: []


**Task 10:** Create a contingency table using `pd.crosstab()` with `region` as rows and `recommended` as columns. Then use `chi2_contingency()` to test for statistical significance. Print the chi-square statistic, p-value, and interpret the results (p < 0.05 indicates significant bias).


In [10]:
# Your code here
region_contingency_table = pd.crosstab(events['region'], events['recommended'])
chi2_type, p_value_type, dof_type, expected_type = chi2_contingency(region_contingency_table)
print(f"\nChi-square statistic: {chi2_type:.4f}")
print(f"P-value: {p_value_type:.4f}")
print(f"Degrees of freedom: {dof_type}")
print(
    f"Is the relationship between region and recommendation significant?: "
    f"{'Yes' if p_value_type < 0.05 else 'No'}" # type: ignore
)

print(f"Region Contingency Table:\n{region_contingency_table}")
print(f"Expected Frequencies:\n{expected_type}")


Chi-square statistic: 389.3530
P-value: 0.0000
Degrees of freedom: 3
Is the relationship between region and recommendation significant?: Yes
Region Contingency Table:
recommended     0     1
region                 
East         5655  6005
North        7328  7776
South        7748  5740
West         5576  4172
Expected Frequencies:
[[6134.7924  5525.2076 ]
 [7946.81856 7157.18144]
 [7096.57632 6391.42368]
 [5128.81272 4619.18728]]


**Task 11:** Group by both `region` and `recommended`, calculating the mean of `converted` and mean of `profit`. Create a pivot table to show these metrics side-by-side for recommended vs not recommended in each region.


In [11]:
# Your code here
events_by_region_and_recommendation = events.groupby(['region', 'recommended']).agg({
    'converted': 'mean',
    'profit': 'mean'
}).reset_index()

pivot_table = (
    events_by_region_and_recommendation
    .pivot(
        index='region',
        columns='recommended',
        values=['converted', 'profit']
    )
    .reset_index()
)

print(f"Pivot Table by Region and Recommendation:\n{pivot_table}")

Pivot Table by Region and Recommendation:
            region converted              profit           
recommended                0         1         0          1
0             East  0.039788  0.090258  3.021162   8.822955
1            North  0.039028  0.097994  3.545364   9.279027
2            South  0.038462  0.090418  3.433458   9.171132
3             West  0.035151  0.095158  2.978117  10.341977


## Age Group Bias Detection

**Task 12:** Group by `age` and calculate recommendation rates. Add columns for difference from average and potential_bias (>10% difference). Sort by age to see patterns.


In [12]:
# Your code here
events_by_age = events.groupby(['age']).agg({
    'recommended': 'mean',
}).reset_index()

overall_recommendation_rate = events['recommended'].mean()

events_by_age['difference_from_average'] = (
    events_by_age['recommended']
    - overall_recommendation_rate
)

events_by_age['potential_bias'] = events_by_age['difference_from_average'].abs() > 0.1

events_by_age.sort_values('age', inplace=True)

print(f"Events by Age:\n{events_by_age}")
print(f"Overall Recommendation Rate: {overall_recommendation_rate}")
print(f"Ages with Potential Bias:\n{events_by_age[events_by_age['potential_bias']]}")

Events by Age:
   age  recommended  difference_from_average  potential_bias
0   18     0.515682                 0.041822           False
1   25     0.534640                 0.060780           False
2   35     0.532656                 0.058796           False
3   45     0.406444                -0.067416           False
4   55     0.402812                -0.071048           False
5   65     0.404771                -0.069089           False
Overall Recommendation Rate: 0.47386
Ages with Potential Bias:
Empty DataFrame
Columns: [age, recommended, difference_from_average, potential_bias]
Index: []


**Task 13:** Create a contingency table for `age` vs `recommended` and perform a chi-square test. Print the results and interpret whether age bias exists.


In [13]:
# Your code here
age_contingency_table = pd.crosstab(events['age'], events['recommended'])
chi2_type, p_value_type, dof_type, expected_type = chi2_contingency(age_contingency_table)
print(f"\nChi-square statistic: {chi2_type:.4f}")
print(f"P-value: {p_value_type:.4f}")
print(f"Degrees of freedom: {dof_type}")
print(
    f"Is the relationship between age and recommendation significant?: "
    f"{'Yes' if p_value_type < 0.05 else 'No'}" # type: ignore
)

print(f"Age Contingency Table:\n{age_contingency_table}")
print(f"Expected Frequencies:\n{expected_type}")


Chi-square statistic: 782.6018
P-value: 0.0000
Degrees of freedom: 5
Is the relationship between age and recommendation significant?: Yes
Age Contingency Table:
recommended     0     1
age                    
18           2517  2680
25           4655  5348
35           5746  6549
45           7442  5096
55           4375  2951
65           1572  1069
Expected Frequencies:
[[2734.34958 2462.65042]
 [5262.97842 4740.02158]
 [6468.8913  5826.1087 ]
 [6596.74332 5941.25668]
 [3854.50164 3471.49836]
 [1389.53574 1251.46426]]


**Task 14:** Filter to converted events only and group by `age` to calculate the average `profit` per conversion for each age group. Sort by profit descending.


In [14]:
# Your code here
events_by_age_and_conversion = (
    events[events['converted']]
    .groupby('age')['profit']
    .mean()
    .reset_index()
    .sort_values('profit', ascending=False)
)

print(f"Events by Age and Conversion:\n{events_by_age_and_conversion}")
print(f"Top Ages by Profit from Conversion:\n{events_by_age_and_conversion.head()}")

Events by Age and Conversion:
   age      profit
5   65  103.958481
3   45  103.294598
1   25   96.518496
2   35   94.999627
4   55   86.578351
0   18   80.184508
Top Ages by Profit from Conversion:
   age      profit
5   65  103.958481
3   45  103.294598
1   25   96.518496
2   35   94.999627
4   55   86.578351


## Customer Segment Analysis

**Task 15:** Filter to converted events and group by `customer_segment`. Calculate the count (num_conversions), mean of `revenue`, and mean of `profit`. Sort by average profit descending.


In [15]:
# Your code here
converted_events = events[events['converted']]

segment_profitability = (
    converted_events
    .groupby('customer_segment')
    .agg(
        num_conversions=('converted', 'count'),
        average_revenue=('revenue', 'mean'),
        average_profit=('profit', 'mean')
    )
    .sort_values('average_profit', ascending=False)
)

print(f"Top Customer Segments by Profitability:\n{segment_profitability.head()}")

Top Customer Segments by Profitability:
                  num_conversions  average_revenue  average_profit
customer_segment                                                  
premium                       621       256.829952      135.381554
regular                      1629       172.871240       91.380913
budget                        975       142.787703       76.269965


**Task 16:** Filter to converted events and group by `customer_segment` to calculate total `profit` for each segment. Add a column showing each segment's percentage of total profit.


In [16]:
# Your code here
segment_total_profit = (
    converted_events
    .groupby('customer_segment')
    .agg(
        total_profit=('profit', 'sum')
    )
)

total_profit = segment_total_profit['total_profit'].sum()

segment_total_profit['profit_percentage'] = (
    segment_total_profit['total_profit']
    / total_profit
) * 100

segment_total_profit = segment_total_profit.sort_values(
    'profit_percentage',
    ascending=False
)

print(f"Top Customer Segments by Total Profit Percentage:\n{segment_total_profit.head()}")

Top Customer Segments by Total Profit Percentage:
                  total_profit  profit_percentage
customer_segment                                 
regular             148859.508          48.441943
premium              84071.945          27.358739
budget               74363.216          24.199319


## Model Version Cost-Effectiveness by Segment

**Task 17:** Group by both `model_version` and `customer_segment`. Calculate sum of `inference_cost` and sum of `profit`. Add an `roi_percentage` column: (profit / inference_cost) × 100. Sort by ROI descending.


In [17]:
# Your code here
segment_model_roi = (
    events
    .groupby(['model_version', 'customer_segment'])
    .agg(
        inference_cost=('inference_cost', 'sum'),
        profit=('profit', 'sum')
    )
    .reset_index()
)

segment_model_roi['roi_percentage'] = (
    segment_model_roi['profit']
    / segment_model_roi['inference_cost']
) * 100

segment_model_roi = segment_model_roi.sort_values(
    'roi_percentage',
    ascending=False
)

print(f"Segment Model ROI:\n{segment_model_roi}")


Segment Model ROI:
  model_version customer_segment  inference_cost     profit  roi_percentage
1          v1.0          premium           4.332  22858.308   527661.772853
2          v1.0          regular          10.050  43250.400   430352.238806
4          v1.1          premium           6.396  23812.424   372301.813634
7          v2.0          premium          11.640  37380.920   321141.924399
5          v1.1          regular          14.600  45345.660   310586.712329
3          v1.1           budget           8.968  26748.642   298267.640500
0          v1.0           budget           6.144  17044.226   277412.532552
8          v2.0          regular          27.660  60216.180   217701.301518
6          v2.0           budget          16.860  30541.330   181146.678529


**Task 18:** Using the previous result, create a pivot table with `model_version` as rows and `customer_segment` as columns, showing `roi_percentage`. Then print which model has the best ROI for each segment.


In [18]:
# Your code here
roi_pivot = (
    segment_model_roi
    .pivot(
        index='model_version',
        columns='customer_segment',
        values='roi_percentage'
    )
)

print(roi_pivot)

best_model_per_segment = roi_pivot.idxmax()

print(f"Best model by customer segment:\n{best_model_per_segment}")

customer_segment         budget        premium        regular
model_version                                                
v1.0              277412.532552  527661.772853  430352.238806
v1.1              298267.640500  372301.813634  310586.712329
v2.0              181146.678529  321141.924399  217701.301518
Best model by customer segment:
customer_segment
budget     v1.1
premium    v1.0
regular    v1.0
dtype: str


## Statistical Bias Testing Across Model Versions

**Task 19:** Filter events to v1.0 and v2.0 separately. For each, create a contingency table of `region` vs `recommended` and run chi-square tests. Compare the p-values to see if regional bias improved or worsened.


In [19]:
# Your code here
v1 = events[events['model_version'] == 'v1.0']
v2 = events[events['model_version'] == 'v2.0']

# v1 regional bias
region_table_v1 = pd.crosstab(
    v1['region'],
    v1['recommended']
)

chi2_v1, p_v1, dof_v1, exp_v1 = chi2_contingency(
    region_table_v1
)

# v2 regional bias
region_table_v2 = pd.crosstab(
    v2['region'],
    v2['recommended']
)

chi2_v2, p_v2, dof_v2, exp_v2 = chi2_contingency(
    region_table_v2
)

print(f"v1.0 Region Bias p-value: {p_v1:.4f}")
print(f"v2.0 Region Bias p-value: {p_v2:.4f}")

if p_v2 > p_v1: # type: ignore
    print("Regional fairness improved in v2.0")
elif p_v2 < p_v1: # type: ignore
    print("Regional fairness worsened in v2.0")
else:
    print("No change detected")

v1.0 Region Bias p-value: 0.0000
v2.0 Region Bias p-value: 0.9637
Regional fairness improved in v2.0


**Task 20:** Similar to Task 19, test for age bias across v1.0 and v2.0 model versions. Compare p-values and calculate the percentage improvement in fairness if v2.0 is better.


In [20]:
# Your code here
age_table_v1 = pd.crosstab(
    v1['age'],
    v1['recommended']
)

chi2_age_v1, p_age_v1, dof_age_v1, exp_age_v1 = (
    chi2_contingency(age_table_v1)
)

age_table_v2 = pd.crosstab(
    v2['age'],
    v2['recommended']
)

chi2_age_v2, p_age_v2, dof_age_v2, exp_age_v2 = (
    chi2_contingency(age_table_v2)
)

print(f"v1.0 Age Bias p-value: {p_age_v1:.4f}")
print(f"v2.0 Age Bias p-value: {p_age_v2:.4f}")

if p_age_v2 > p_age_v1: # type: ignore

    fairness_improvement = (
        (p_age_v2 - p_age_v1) # type: ignore
        / p_age_v1
    ) * 100

    print(
        f"Fairness improved by {fairness_improvement:.2f}%"
    )

else:
    print("No fairness improvement detected")

v1.0 Age Bias p-value: 0.6328
v2.0 Age Bias p-value: 0.0000
No fairness improvement detected


## Final Recommendations and Summary

**Task 21:** Create a comprehensive summary report. For all three model versions, calculate: conversion rate, ROI, total profit, region bias p-value, and age bias p-value. Add a column indicating if the model has significant bias (p < 0.05 for either region or age).


In [21]:
# Your code here
summary_rows = []

for model in sorted(events['model_version'].unique()):

    model_data = events[
        events['model_version'] == model
    ]

    converted_model_data = model_data[
        model_data['converted']
    ]

    # --------------------------------------------------
    # Performance Metrics
    # --------------------------------------------------

    conversion_rate = model_data['converted'].mean()

    recommendation_rate = model_data['recommended'].mean()

    total_revenue = model_data['revenue'].sum()

    total_profit = model_data['profit'].sum()

    total_inference_cost = model_data['inference_cost'].sum()

    total_product_cost = model_data['product_cost'].sum()

    ai_roi = (
        total_profit
        / total_inference_cost
    ) * 100

    overall_roi = (
        total_profit
        /
        (total_inference_cost + total_product_cost)
    ) * 100


    # --------------------------------------------------
    # Region Bias
    # --------------------------------------------------

    region_table = pd.crosstab(
        model_data['region'],
        model_data['recommended']
    )

    _, region_p_value, _, _ = chi2_contingency(
        region_table
    )


    # --------------------------------------------------
    # Age Bias
    # --------------------------------------------------

    age_table = pd.crosstab(
        model_data['age'],
        model_data['recommended']
    )

    _, age_p_value, _, _ = chi2_contingency(
        age_table
    )


    # --------------------------------------------------
    # Customer Segment Profitability
    # (Task 15)
    # --------------------------------------------------

    segment_stats = (
        converted_model_data
        .groupby('customer_segment')
        .agg(
            num_conversions=('converted', 'count'),
            avg_revenue=('revenue', 'mean'),
            avg_profit=('profit', 'mean'),
            total_profit=('profit', 'sum')
        )
    )

    top_profit_segment = (
        segment_stats['total_profit']
        .idxmax()
    )

    top_profit_value = (
        segment_stats['total_profit']
        .max()
    )

    top_avg_profit_segment = (
        segment_stats['avg_profit']
        .idxmax()
    )

    top_avg_profit_value = (
        segment_stats['avg_profit']
        .max()
    )


    # --------------------------------------------------
    # Customer Segment Contribution
    # (Task 16)
    # --------------------------------------------------

    segment_share = (
        segment_stats['total_profit']
        /
        segment_stats['total_profit'].sum()
    ) * 100

    largest_profit_share = segment_share.max()

    largest_profit_share_segment = (
        segment_share.idxmax()
    )


    # --------------------------------------------------
    # Segment ROI
    # (Task 17)
    # --------------------------------------------------

    segment_roi = (
        model_data
        .groupby('customer_segment')
        .agg(
            inference_cost=('inference_cost', 'sum'),
            profit=('profit', 'sum')
        )
    )

    segment_roi['roi_percentage'] = (
        segment_roi['profit']
        / segment_roi['inference_cost']
    ) * 100

    best_roi_segment = (
        segment_roi['roi_percentage']
        .idxmax()
    )

    best_roi_value = (
        segment_roi['roi_percentage']
        .max()
    )


    summary_rows.append({

        'model_version': model,

        'conversion_rate':
            conversion_rate,

        'recommendation_rate':
            recommendation_rate,

        'total_revenue':
            total_revenue,

        'total_profit':
            total_profit,

        'ai_roi':
            ai_roi,

        'overall_roi':
            overall_roi,

        'region_bias_p_value':
            region_p_value,

        'age_bias_p_value':
            age_p_value,

        'region_bias':
            region_p_value < 0.05, # type: ignore

        'age_bias':
            age_p_value < 0.05, # type: ignore

        'significant_bias':
            (
                region_p_value < 0.05  # type: ignore
                or
                age_p_value < 0.05  # type: ignore
            ),

        'top_profit_segment':
            top_profit_segment,

        'top_profit_segment_total_profit':
            top_profit_value,

        'top_avg_profit_segment':
            top_avg_profit_segment,

        'top_avg_profit_per_conversion':
            top_avg_profit_value,

        'largest_profit_share_segment':
            largest_profit_share_segment,

        'largest_profit_share_pct':
            largest_profit_share,

        'best_roi_segment':
            best_roi_segment,

        'best_roi_segment_roi':
            best_roi_value

    })

summary_report = pd.DataFrame(summary_rows)

summary_report = summary_report.sort_values(
    'overall_roi',
    ascending=False
)

print(summary_report)

  model_version  conversion_rate  recommendation_rate  total_revenue  \
2          v2.0         0.065766             0.459725      241108.87   
1          v1.1         0.066712             0.504784      180973.53   
0          v1.0         0.060541             0.462297      158234.26   

   total_profit         ai_roi  overall_roi  region_bias_p_value  \
2    128138.430  228166.720085   113.426512         9.636543e-01   
1     95906.726  320073.174476   112.742834         8.731995e-01   
0     83152.934  405110.269902   110.750487        3.627345e-287   

   age_bias_p_value  region_bias  age_bias  significant_bias  \
2          0.000000        False      True              True   
1          0.234890        False     False             False   
0          0.632756         True     False              True   

  top_profit_segment  top_profit_segment_total_profit top_avg_profit_segment  \
2            regular                        60241.188                premium   
1            regular 

## Your Recommendations

Based on your analysis, consider the following questions:

1. **Which model version delivers the best ROI?** Look at Task 7 results.

2. **Are there statistically significant biases?** Check p-values from Tasks 10, 13, 19, and 20.

3. **Which customer segments are most profitable?** Review Tasks 15-16.

4. **Should you use different models for different segments?** Examine Task 18.

5. **Did fairness improve with newer models?** Compare Tasks 19 and 20.

6. **What's your deployment recommendation?** Consider:
   - Deploy one model everywhere?
   - Use smart routing (different models for different segments)?
   - Roll back to v1.0 due to bias concerns?
   - Invest in v2.1 to address identified biases?

Write a 2-3 paragraph executive summary with your recommendations.


# Executive Summary

Analysis of the three recommendation models reveals a tradeoff between profitability, efficiency, and fairness. **Model v2.0** generated the highest total profit ($128.1K) and achieved the strongest overall ROI (113.4%), making it the **top performer from a pure business revenue perspective**. **Model v1.1** delivered the **highest conversion rate** (6.67%), while **model v1.0** produced the **highest AI-specific ROI** due to its lower inference costs. Although all three models generated positive returns, v2.0 appears to maximize business value, while v1.0 maximizes efficiency.

Fairness analysis produced a different ranking. Statistical testing found significant regional bias in v1.0 and significant age bias in v2.0. Only v1.1 showed no statistically significant evidence of either regional or age bias. **From a governance and responsible AI perspective, v1.1 is therefore the strongest candidate for broad deployment**. This finding is especially important because v2.0's profitability gains come with measurable age-based fairness concerns, while v1.0 exhibits extremely strong regional bias.

Customer segment analysis showed that regular customers generate the largest share of overall profit, contributing approximately half of all profits across every model. However, premium customers consistently deliver the highest profit per conversion and the highest ROI, suggesting that recommendation investments are most effective in that segment. **Based on the available evidence, the recommended strategy is to deploy v1.1 as the default production model because it provides strong conversion performance while remaining free of statistically significant bias**. If the organization is willing to accept additional fairness risk in exchange for higher profit, v2.0 could be considered after age-bias mitigation work is completed. A future v2.1 release should focus specifically on preserving v2.0's profitability while addressing its age-related fairness concerns.